In [2]:
import numpy as np
import scanpy as sc
import pandas as pd
from model import test
from tool.matrix import adj_mapping_matrix
from tool.spatial_df import spatial_graph
from tool.pseudo_df import pseudo_cal_adj
from tool.deconv_metric import CalDataMetric
from tool.data import mapping_adj2matrix,adata_to_cluster_expression

In [112]:
DataDir="./data/Exseq/"
sc_file_path = DataDir+'scRNA.h5ad'
spatial_file_path = DataDir+'Spatial.h5ad'
pseudo_file_path = DataDir+'Pseudo.h5ad'
RNA_data_adata = sc.read_h5ad(sc_file_path)
Spatial_data_adata = sc.read_h5ad(spatial_file_path)
pseudo_data_adata=sc.read_h5ad(pseudo_file_path)

RNA_data_adata = sc.read_h5ad(sc_file_path)
Spatial_data_adata = sc.read_h5ad(spatial_file_path)
# RNA_data_adata.obs['celltype']=RNA_data_adata.obs['celltype_final']
celltype_counts = RNA_data_adata.obs['celltype'].value_counts()
print(celltype_counts)
print(Spatial_data_adata)
print(pseudo_data_adata)
print(RNA_data_adata)

Excitatory L6       3190
Excitatory L5       1786
Inhibitory Sst      1741
Inhibitory Vip      1728
Excitatory L4       1401
Inhibitory Pvalb    1337
Inhibitory Other    1274
Excitatory L2/3      982
Astro                368
Other                 99
Endo                  94
Olig                  91
Smc                   55
Neuron Other          52
Micro                 51
Name: celltype, dtype: int64
AnnData object with n_obs × n_vars = 1179 × 34041
    obs: 'X', 'Y', 'n_genes'
    var: 'n_cells'
    uns: 'density'
AnnData object with n_obs × n_vars = 298 × 34041
    obs: 'X', 'Y'
    uns: 'density'
AnnData object with n_obs × n_vars = 14249 × 34041
    obs: 'celltype'


In [113]:
RNA_ret_adata=RNA_data_adata.copy()
RNA_data_adata2=RNA_data_adata.copy()
sc.pp.normalize_total(RNA_ret_adata)
RNA_ret_adata=adata_to_cluster_expression(RNA_ret_adata)

In [114]:
gd_results=Spatial_data_adata.uns['density']
gd_results = gd_results.loc[:,np.unique(gd_results.columns)]
gd_results = (gd_results.T/gd_results.sum(axis=1)).T
gd_results = gd_results.fillna(0)

pseudo_results=pseudo_data_adata.uns['density']
pseudo_results = pseudo_results.loc[:,np.unique(pseudo_results.columns)]
pseudo_results = (pseudo_results.T/pseudo_results.sum(axis=1)).T
pseudo_results = pseudo_results.fillna(0)

print(gd_results)
print(pseudo_results)

      Astro  Endo  Excitatory L2/3  Excitatory L4  Excitatory L5  \
0       0.0   0.0         0.000000       0.600000       0.100000   
1       0.0   0.0         0.090909       0.000000       0.000000   
2       0.0   0.0         0.082284       0.139606       0.068576   
3       0.0   0.1         0.000000       0.000000       0.100000   
4       0.0   0.0         0.000000       0.000000       0.000000   
...     ...   ...              ...            ...            ...   
1174    0.0   0.0         0.000000       0.000000       0.000000   
1175    0.0   0.0         0.078056       0.060917       0.107840   
1176    0.0   0.0         0.067618       0.101293       0.050838   
1177    0.0   0.0         0.086166       0.135790       0.064411   
1178    0.0   0.0         0.125000       0.000000       0.000000   

      Excitatory L6  Inhibitory Other  Inhibitory Pvalb  Inhibitory Sst  \
0          0.000000          0.100000          0.200000        0.000000   
1          0.090909          0.00

In [20]:
adj_df=spatial_graph(RNA_ret_adata,Spatial_data_adata,DataDir,q=20, p=20)

Data exists, load it
load adj dataframe


In [22]:
pseudo_df=pseudo_cal_adj(RNA_ret_adata,pseudo_data_adata,DataDir, p=20)

Data exists, load it
load adj dataframe


In [ ]:
concat_st_sc,concat_pseudo_sc=data_preprocessing(RNA_data_adata,Spatial_data_adata,pseudo_data_adata)

In [24]:
adj_matrix=adj_mapping_matrix(adj_df,concat_st_sc)
pseudo_matrix=adj_mapping_matrix(pseudo_df,concat_pseudo_sc)

In [ ]:
ad_map = test.start(
    concat_st_sc=concat_st_sc,
    concat_pseudo_sc=concat_pseudo_sc,
    adj=adj_matrix,
    pseudo_adj=pseudo_matrix,
    gd_results=gd_results,
    pseudo_results=pseudo_results,
    α=0.1,
    β=1.0,
    device='cuda',
    seed=2010,
    gcn_hidden_dims=[2048]
)

In [42]:
ad_map.obs=ad_map.obs.dropna(axis=1, how='all')
ad_map.var=ad_map.var.dropna(axis=1, how='all')
print(ad_map)
save_path=DataDir+'result/GCIRS.h5ad'
ad_map.write(save_path)

AnnData object with n_obs × n_vars = 1179 × 15
    obs: 'X', 'Y', 'n_genes', 'rna_count_based_density', 'batch'
    var: 'celltype', 'cluster_density', 'batch'
